In [24]:
using Oscar
using LinearAlgebra
using HomotopyContinuation
using Plots

# PRE: V is a collection of vertices in R^d defining a polytope.
# POST: Returns the d-dimensional volume of the polytope spanned by V.
function areaofpolytope(V)
    P = convex_hull(QQ,V)
    return volume(P)
end

#W = [[1,0,0], [1,1,0], [0,1,0], [0,0,0], [1,0,1], [1,1,1], [0,1,1], [0,0,1]]
W = [
    [QQ(-2,1),   QQ(-3,2),   QQ(4,5)],
    [QQ(5,2),    QQ(-4,5),   QQ(4,5)],
    [QQ(-3,10),  QQ(19,10),  QQ(4,5)],
    [QQ(-2,1),   QQ(-3,2),   QQ(1,1)],
    [QQ(5,2),    QQ(-4,5),   QQ(1,1)],
    [QQ(-3,10),  QQ(19,10),  QQ(1,1)]
]
V = [[QQ(v) for v in w] for w in W]
areaofpolytope(V)

1411//1000

In [25]:
# PRE: V is a collection of vertices in R^3 defining a polytope.
# POST: Returns the lifted vertices in R^4, where each vertex v is mapped
#       to (v,-1).
function vertex_embed(V)
    embedding = Vector{Vector{QQFieldElem}}()
    for v in V
        #println(v)
        v_lift = copy(v)
        #println(v_lift)
        push!(v_lift,-1)
        push!(embedding,v_lift)
    end
    return embedding
end
println(vertex_embed(V))
println("V=",V)

Vector{QQFieldElem}[[-2, -3//2, 4//5, -1], [5//2, -4//5, 4//5, -1], [-3//10, 19//10, 4//5, -1], [-2, -3//2, 1, -1], [5//2, -4//5, 1, -1], [-3//10, 19//10, 1, -1]]
V=Vector{QQFieldElem}[[-2, -3//2, 4//5], [5//2, -4//5, 4//5], [-3//10, 19//10, 4//5], [-2, -3//2, 1], [5//2, -4//5, 1], [-3//10, 19//10, 1]]


In [26]:
# PRE: v is a point in R^3 and h is a vector of coefficients defining the
#      hyperplane H = {x : h · (x,-1) = 0}.
# POST: Returns the evaluation of the hyperplane equation at v, i.e.,
#       h · (v,-1).
function inner_product(v,h)
    S = parent(v[1])
    v_lift = [S(x) for x in v]
    push!(v_lift, -S(1))
    s = sum(h[i]*v_lift[i] for i in 1:length(h))
    return s
end

for v in V
    println(inner_product(v, [1,1,1,1]))
end

-37//10
3//2
7//5
-7//2
17//10
8//5


In [27]:
# PRE: V is a collection of vertices in R^3 defining a polytope, and h is a
#      vector of coefficients defining the hyperplane
#      H = {x : h · (x,-1) = 0}.
# POST: Returns a vector containing the 2-dimensional faces of the polytope
#       that are intersected by the hyperplane H.
function hyperplane_intersect(V,h)
	P = convex_hull(QQ,V)
	cut = []
    
	for e in faces(P, Oscar.ambient_dim(P)-1)
        #println(e)
		s = [inner_product(collect(v),h) for v in vertices(e)]
        t = 0
        #find the first non-zero value in the vector s
        for i in 1:length(s)
            if s[i] != 0
                t = s[i]
                break
            end
        end
        is_cut = false
        #if there is some value on s of opposite sign as t, push the face into cut
		for r in s
            if (r != 0 && sign(r) != sign(t))
                is_cut = true
                break
            end
        end
        if is_cut
            push!(cut,e)
        end
	end
	return cut #Chiara: this now outputs two edges instead of just the indices of the corresponding vertices
end

edges = hyperplane_intersect(V, [1,1,1,1])
for e in edges
    println(e)
end


Polytope in ambient dimension 3
Polytope in ambient dimension 3
Polytope in ambient dimension 3
Polytope in ambient dimension 3


In [28]:
# PRE: V is a collection of vertices in R^3 defining a polytope, and h is a
#      vector of coefficients defining the hyperplane H = {x : h · (x,-1) = 0}.
# POST: Returns a vector containing the points where H intersects the edges of
#       the polytope spanned by V.
function hyperplane_intersectpoints(V,h)
    facets = hyperplane_intersect(V,h)
    newpoints = Vector{Vector{QQFieldElem}}()
    points = Vector{Vector{QQFieldElem}}()
    for facet in facets
        for f in faces(facet,Oscar.ambient_dim(facet)-2)
            v = [collect(vv) for vv in vertices(f)]
            s1 = inner_product(v[2],h)
            s2 = inner_product(v[1],h)
            if s1 * s2 <= 0 && !(s1 == 0 && s2 == 0)
                l = s1 //(s1 - s2)
                p = [l*v[1][i] + (1-l)*v[2][i] for i in 1:length(v[1])]
                push!(points, p)
            end
        end
    end
    for p in points
        if !(p in newpoints)
            push!(newpoints,p)
        end
    end
return newpoints
end

hyperplane_intersectpoints(V,[1,1,1,3//2])

4-element Vector{Vector{QQFieldElem}}:
 [85//52, -243//260, 4//5]
 [19//13, -25//26, 1]
 [-3//5, 13//10, 4//5]
 [-2//3, 7//6, 1]

In [29]:
# PRE: V is a collection of vertices in R^d defining a polytope, and h is a
#      vector of coefficients defining the hyperplane H = {x : h · (x,-1) = 0}.
# POST: Returns the 3-dimensional volume of P(h < 0).
function negative_areaofcut(V,h)
    newpoints = hyperplane_intersectpoints(V,h)
    points = Vector{Vector{QQFieldElem}}()
    for v in V
        s = inner_product(v,h)
        if s < 0
            push!(points, v)        
        end
    end
    for v in newpoints
        push!(points, v)
    end    
    Pplus = convex_hull(QQ, points)
    return(volume(Pplus))
end   

Float64(negative_areaofcut(V,[1,1,1,1]))

negative_areaofcut(V,[0,4,4,2])

48389//81000

In [30]:
# PRE: V is a collection of vertices in R^d defining a polytope.
# POST: Returns the maximal open chambers of the hyperplane
#       arrangement induced by the lifted vertices of V.
function get_chambers(V)
    n = length(V)
    embed = vertex_embed(V)
    ha = transpose(reduce(hcat, embed))
    HA = Polymake.fan.HyperplaneArrangement(HYPERPLANES=ha)
    CD = HA.CHAMBER_DECOMPOSITION
    
    fan = polyhedral_fan(CD)
    chambers = maximal_cones(fan)
    return chambers
end
#get_chambers(V)

get_chambers (generic function with 1 method)

In [32]:
# PRE: V is a collection of vertices in R^3 defining a polytope.
# POST: Returns a representative vector for each maximal chamber of the
#       hyperplane arrangement whose corresponding hyperplane intersects the
#       polytope in a non-empty 2-dimensional section.
function representing_vectors(V)
    chambers = get_chambers(V)
    reprvec = []
    for ch in chambers
        u = sum(rays(ch))
        facecut = hyperplane_intersect(V, u)
        if length(facecut) > 2
            push!(reprvec, (chamber = ch, repr = collect(QQFieldElem, u)))
        end
    end
    return reprvec
end


println(representing_vectors(V))

Any[(chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[5, -2605//378, 0, 9179//945]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[3, -2227//378, 87//4, 56347//1890]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[3, -2227//378, -87//4, -35293//3780]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[4, 29//27, 5063//108, 7649//180]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[4, 29//27, -5063//108, -377//9]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[3, 1139//126, 0, -34427//3780]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[2, 3025//378, 112391//756, 86869//630]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[2, 3025//378, -112391//756, -32687//252]), (chamber = Polyhedral cone in ambient dimension 4, repr = QQFieldElem[-1, 3011//378, -1425//14, -325651//3780]), (chamber = Polyhedral cone in 

In [33]:
# PRE: V is a collection of vertices in R^3 defining a polytope, and h is a
#      representative vector defining the hyperplane
#      H = {x : h · (x,-1) = 0}.
# POST: Returns the numerical and symbolic coordinates of the vertices of the
#       polytope obtained by cutting conv(V) with H. The symbolic coordinates are
#       rational functions in the hyperplane coefficients.
function symbolic_intersection(V,h)
    n = length(V[1])
    variables = ["a$i" for i in 1:n+1]
    P = convex_hull(QQ,V)
    R, vars = polynomial_ring(QQ, variables)
    S = fraction_field(R)

    c = [S(v) for v in vars]
    #this gives the facets getting cut by a hyperplane with coefficients lying in the chamber represented by h
    
    points_num = Vector{Vector{QQFieldElem}}()
    points_sym = Vector{Vector{eltype(S)}}()

     for v in V
        s = inner_product(v,h) 
        if s < 0 #what about = 0?
            push!(points_num, v)
            push!(points_sym, [S(x) for x in v])
        end
    end
        
    for f in faces(P, 1)
        v = [collect(vv) for vv in vertices(f)]
        s1_num = inner_product(v[2],h)
        s2_num = inner_product(v[1],h)

        s1 = inner_product([S(w) for w in v[2]],c)
        s2 = inner_product([S(w) for w in v[1]],c)
        
        if s1_num * s2_num <= 0 && !(s1_num == 0 && s2_num == 0)
            l_num = s1_num //(s1_num - s2_num)
            l_sym = s1 //(s1 - s2)
            
            p_sym = [l_sym*S(v[1][i]) + (1-l_sym)*S(v[2][i]) for i in 1:length(v[1])]
            p_num = [l_num*v[1][i] + (1-l_num)*v[2][i] for i in 1:length(v[1])]
                            
            push!(points_num, p_num)
            push!(points_sym,p_sym)
        end
    end

    return points_num, points_sym, S
end

reprvec = representing_vectors(V)
println(reprvec[2].repr)
symbolic_intersection(V, reprvec[2].repr)

QQFieldElem[3, -2227//378, 87//4, 56347//1890]


(Vector{QQFieldElem}[[-2, -3//2, 4//5], [5//2, -4//5, 4//5], [-3//10, 19//10, 4//5], [-2, -3//2, 1], [-3//10, 19//10, 1], [5//2, -4//5, 352//435], [31//61, -677//610, 1], [829//410, -139//410, 1]], Vector{Any}[[-2, -3//2, 4//5], [5//2, -4//5, 4//5], [-3//10, 19//10, 4//5], [-2, -3//2, 1], [-3//10, 19//10, 1], [5//2, -4//5, (-5//2*a1 + 4//5*a2 + a4)//a3], [(107//90*a2 - a3 + a4)//(a1 + 7//45*a2), (-107//90*a1 - 7//45*a3 + 7//45*a4)//(a1 + 7//45*a2), 1], [(-451//280*a2 - a3 + a4)//(a1 - 27//28*a2), (451//280*a1 + 27//28*a3 - 27//28*a4)//(a1 - 27//28*a2), 1]], Fraction field of multivariate polynomial ring)

In [34]:
# PRE: sx is a list of vertices defining a simplex, and S is the fraction field
#      containing the coordinates of the simplex vertices.
# POST: Returns the volume of the simplex using the determinant formula.
function vol_simplex(sx,S)
    n = length(sx)
    columns = [sx[i] - sx[1] for i in 2:n]
    M = reduce(hcat, columns)
    #println(M)
    M_matrix = matrix(S, M)
    volume = Oscar.det(M_matrix)//factorial(n-1)
    return volume
end

# R, a = polynomial_ring(QQ, :a => 1:2)
# S = fraction_field(R)
# T = [[0, 1,1], [QQ(1),QQ(0),QQ(0)], [QQ(0),QQ(1),QQ(0)], [QQ(0),QQ(0),QQ(1)]]

# vol_simplex(T,S)  

vol_simplex (generic function with 1 method)

In [35]:
# PRE: V is a set of vertices defining a polytope in R^3, and h is a
#      representative vector of a chamber in coefficient space.
# POST: Returns a pair containing the numerical volume of the cut polytope for
#       h and the symbolic rational function giving the volume for every
#       hyperplane whose coefficient vector lies in the same chamber as h.
function local_volume(V,h)
    #now we take a look at the points numerically and symbolically
    points_num, points_sym, S = symbolic_intersection(V,h)
    #create a convex hull of the numerical intersection using the representative and take its triangulation 
    Pplus = convex_hull(QQ, points_num)
    #this gives me the indices of the vertices needed
    triang = regular_triangulation(Pplus)

    #now to compute a rational volume function, we create a list of vertices, where the representative intersection 
    #points are substituted by the corresponding functions
    volume_sym = S(0)
    volume_num = QQ(0)
    for index in triang[1]
        sx_num = points_num[index]
        sx_sym = points_sym[index]
        #println(sx)
        volume_num += vol_simplex(sx_num, QQ)
        #volume_sym += vol_simplex(sx_sym, S)
        volume_sym += sign(vol_simplex(sx_num, QQ))*vol_simplex(sx_sym, S)
    end
    volume = (num = volume_num, sym = volume_sym)
    return volume
end

println(local_volume(V, reprvec[2].repr))

(num = 54858269//108793500, sym = (-35275//12096*a1^3 + 1411//504*a1^2*a2 - 526303//252000*a1^2*a3 + 7055//2016*a1^2*a4 - 1411//1575*a1*a2^2 + 153799//140000*a1*a2*a3 - 1411//630*a1*a2*a4 - 1411//1008*a1*a3^2 + 1411//504*a1*a3*a4 - 1411//1008*a1*a4^2 + 11288//118125*a2^3 - 718199//1260000*a2^2*a3 + 2822//7875*a2^2*a4 + 1411//3150*a2*a3^2 - 1411//1575*a2*a3*a4 + 1411//3150*a2*a4^2 - 1411//7560*a3^3 + 1411//2520*a3^2*a4 - 1411//2520*a3*a4^2 + 1411//7560*a4^3)//(a1^2*a3 - 1019//1260*a1*a2*a3 - 3//20*a2^2*a3))


In [36]:
# PRE: V is a collection of vertices in R^d defining a polytope.
# POST: For each chamber of the hyperplane arrangement, returns a rational
#       function describing the volume of the cut polytope for hyperplanes
#       whose coefficient vector lies in that chamber.
function volume_functions(V)
    reprvec = representing_vectors(V)
    functions = []

    for item in reprvec
        h = item.repr
        ch = item.chamber
        
        vol = local_volume(V,h)
        vol_num = vol.num
        vol_sym = vol.sym
        volume = sign(vol_num)*vol_sym
        
        push!(functions, (chamber = ch, repr = h, local_vol = volume))
    end
    return functions
end

volume_functions (generic function with 1 method)

In [38]:
# PRE: V is a set of vertices defining a polytope P ⊂ R^d.
# POST: For every chamber, returns the corresponding chamber, a representative
#       vector, and the polynomial equation obtained from the condition that
#       the local volume function equals half of the total volume of P.
function halving_equations(V)
    n = length(V[1])
    variables = ["a$i" for i in 1:n+1]
    R, vars = polynomial_ring(QQ, variables)
    S = fraction_field(R)

    vol_loc = volume_functions(V)
    area = areaofpolytope(V)
    equations = []
    
    for tuple in vol_loc
        ch = tuple.chamber
        fct = tuple.local_vol
        p = numerator(fct)
        q = denominator(fct)
        f = 2*p - area*q
        push!(equations,(chamber = ch, repr= tuple.repr, volume_ch = f ))
    end
    return equations
end

for i in 1:length(halving_equations(V))
    println(halving_equations(V)[i].volume_ch)
end

-356983//126000*a1^2 + 3184627//1260000*a1*a2 + 1411//1400*a1*a3 - 1411//1260*a1*a4 + 196129//252000*a2^2 - 1411//4375*a2*a3 + 2822//7875*a2*a4 + 86071//472500*a3^2 - 1411//3500*a3*a4 + 1411//6300*a4^2
-35275//6048*a1^3 + 1411//252*a1^2*a2 - 704089//126000*a1^2*a3 + 7055//1008*a1^2*a4 - 2822//1575*a1*a2^2 + 4206191//1260000*a1*a2*a3 - 1411//315*a1*a2*a4 - 1411//504*a1*a3^2 + 1411//252*a1*a3*a4 - 1411//504*a1*a4^2 + 22576//118125*a2^3 - 1169719//1260000*a2^2*a3 + 5644//7875*a2^2*a4 + 1411//1575*a2*a3^2 - 2822//1575*a2*a3*a4 + 1411//1575*a2*a4^2 - 1411//3780*a3^3 + 1411//1260*a3^2*a4 - 1411//1260*a3*a4^2 + 1411//3780*a4^3
35275//6048*a1^3 - 1411//252*a1^2*a2 + 441643//63000*a1^2*a3 - 7055//1008*a1^2*a4 + 2822//1575*a1*a2^2 - 5953009//1260000*a1*a2*a3 + 1411//315*a1*a2*a4 + 2822//1575*a1*a3^2 - 1411//315*a1*a3*a4 + 1411//504*a1*a4^2 - 22576//118125*a2^3 + 455753//1260000*a2^2*a3 - 5644//7875*a2^2*a4 - 22576//39375*a2*a3^2 + 11288//7875*a2*a3*a4 - 1411//1575*a2*a4^2 + 22576//118125*a3^3 - 

LoadError: InterruptException:

In [39]:
# PRE: f is a polynomial in Oscar with variables vars, i.e. f in k[vars].
# POST: Returns the equivalent polynomial expression in the format required by
#       HomotopyContinuation.
function oscar_to_homcon(f,vars)
    result = HomotopyContinuation.Expression(0)
    n = length(vars)
    for t in terms(f)
        c = collect(AbstractAlgebra.coefficients(t))
        e = collect(AbstractAlgebra.exponent_vectors(t))
        monom = prod(vars[i]^e[1][i] for i in 1:n)
        coeff = numerator(c[1])//denominator(c[1])
        term = Rational(coeff)*monom
        result += term
    end
    return result
end

oscar_to_homcon (generic function with 1 method)

In [40]:
# PRE: Inputs are three polytopes P, Q, and T in R^3.
# POST: Returns the hyperplane coefficient vectors whose corresponding
#       hyperplanes bisect all three polytopes simultaneously.
function ham_sandwich(P,Q,T)

    V = [collect(v) for v in vertices(P)]
    W = [collect(v) for v in vertices(Q)]
    X = [collect(v) for v in vertices(T)]
    
    n = length(V[1])
    if (length(V) < n || length(W) < n || length(X) < n)
        println("One of the Polygons is degenerate")
        return
    end
    #vogliamo risolvere il sistema di soluzioni usando il metodo dato da homotopycontinuation. Per fare ciò, è importante riuscire a convertire le funzioni da QQMPolyElem a Expression
    Lp = halving_equations(V)
    Lq = halving_equations(W)
    Lt = halving_equations(X)

    half_V= Float64(areaofpolytope(V))/2
    half_W= Float64(areaofpolytope(W))/2
    half_X= Float64(areaofpolytope(X))/2

    @var a b c d
    vars = [a,b,c,d]
    solutions = []

    for p_tuple in Lp
        cp = p_tuple.chamber
        fp = p_tuple.volume_ch
        fp_hc = oscar_to_homcon(fp, vars)
        isempty(variables(fp_hc)) && continue
        
        for q_tuple in Lq
            cq = q_tuple.chamber
            fq = q_tuple.volume_ch
            fq_hc = oscar_to_homcon(fq, vars)
            isempty(variables(fq_hc)) && continue
            
            for t_tuple in Lt
                ct = t_tuple.chamber
                ft = t_tuple.volume_ch
                ft_hc = oscar_to_homcon(ft, vars)
                isempty(variables(ft_hc)) && continue
    
                cc = intersect(intersect(cp, cq), ct)

                if Oscar.dim(cc) > n
                    F = HomotopyContinuation.solve([fp_hc, fq_hc, ft_hc, a^2 + b^2 +c^2 - 1])
                    points = HomotopyContinuation.real_solutions(F) 
                    in_cc = [v for v in points if v in cc]
                    for v in in_cc
                        u = [QQ(rationalize(BigFloat(x))) for x in v]
                        if (abs(Float64(negative_areaofcut(V,u)) - half_V) < 1e-6 &&
                            abs(Float64(negative_areaofcut(W,u)) - half_W) < 1e-6 &&
                            abs(Float64(negative_areaofcut(X,u)) - half_X) < 1e-6 )
                            push!(solutions, v)
                        end
                    end
                end
            end
        end
    end
    return solutions
end

ham_sandwich (generic function with 1 method)

In [41]:
# ===== Bread: irregular triangle-cut slab =====
V_bread = [
    [QQ(-2,1),   QQ(-3,2),   QQ(4,5)],
    [QQ(5,2),    QQ(-4,5),   QQ(4,5)],
    [QQ(-3,10),  QQ(19,10),  QQ(4,5)],
    [QQ(-2,1),   QQ(-3,2),   QQ(1,1)],
    [QQ(5,2),    QQ(-4,5),   QQ(1,1)],
    [QQ(-3,10),  QQ(19,10),  QQ(1,1)]
]

# ===== Cream cheese: thin layer, inset ~0.2 units, non-uniform scaling from bread =====
V_cheese = [
    [QQ(-8,5),   QQ(-33,25), QQ(101,100)],
    [QQ(9,4),    QQ(-4,5),   QQ(101,100)],
    [QQ(-9,25),  QQ(42,25),  QQ(101,100)],
    [QQ(-8,5),   QQ(-33,25), QQ(27,25)],
    [QQ(9,4),    QQ(-4,5),   QQ(27,25)],
    [QQ(-9,25),  QQ(42,25),  QQ(27,25)]
]

# ===== Smoked salmon: triangular base, irregular hand-set upper vertices =====
V_salmon = [
    [QQ(-3,2),   QQ(-23,20), QQ(28,25)],
    [QQ(21,10),  QQ(-1,2),   QQ(28,25)],
    [QQ(-1,4),   QQ(31,20),  QQ(28,25)],
    [QQ(-23,20), QQ(-11,20), QQ(131,100)],
    [QQ(3,4),    QQ(-17,20), QQ(31,25)],
    [QQ(1,20),   QQ(3,4),    QQ(27,20)]
]
P = convex_hull(QQ, V_bread)
Q = convex_hull(QQ, V_salmon)
T = convex_hull(QQ, V_cheese)
@time sol = ham_sandwich(P,Q, T)

Tracking 144 paths... 100%|████████████████████████████████████████| Time: 0:00:00
                   # paths tracked: 144
   # non-singular solutions (real): 144 (40)
       # singular endpoints (real): 0 (0)
          # total solutions (real): 144 (40)
3139.640012 seconds (2.05 G allocations: 55.761 GiB, 5.99% gc time, 13.73% compilation time: <1% of which was recompilation)


1-element Vector{Any}:
 [0.5684274333163548, 0.8157935855924692, 0.10663526039525499, 0.15435013300364]

In [16]:
V1 = [
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
    ]
    
# V2 = [
#     [1, 0, 0],
#     [2, 0, 0],
#     [1, 1, 0],
#     [1, 0, 1]
#     ]

V2 = [
    [QQ(0), QQ(0), QQ(0)],
    [QQ(1), QQ(0), QQ(0)],
    [QQ(1), QQ(1), QQ(0)],
    [QQ(0), QQ(1), QQ(0)],
    [QQ(0), QQ(0), QQ(1)],
    [QQ(1), QQ(0), QQ(1)],
    [QQ(1), QQ(1), QQ(1)],
    [QQ(0), QQ(1), QQ(1)]
]


# V3 = [
#     [0, 1, 0],
#     [1, 1, 0],
#     [0, 2, 0],
#     [0, 1, 1]
#     ]

V3 = [
    [QQ(2), QQ(0), QQ(0)],
    [QQ(4), QQ(1), QQ(0)],
    [QQ(3), QQ(3), QQ(1)],
    [QQ(1), QQ(2), QQ(2)],
    [QQ(0), QQ(1), QQ(1)],
    [QQ(1), QQ(0), QQ(2)]
]
    

P = convex_hull(QQ, V1)
Q = convex_hull(QQ, V2)
T = convex_hull(QQ, V3)
@time sol = ham_sandwich(P,Q, T)

LoadError: InterruptException:

In [43]:
function main()
    V_bread = [
        [QQ(-2,1),   QQ(-3,2),   QQ(4,5)],
        [QQ(5,2),    QQ(-4,5),   QQ(4,5)],
        [QQ(-3,10),  QQ(19,10),  QQ(4,5)],
        [QQ(-2,1),   QQ(-3,2),   QQ(1,1)],
        [QQ(5,2),    QQ(-4,5),   QQ(1,1)],
        [QQ(-3,10),  QQ(19,10),  QQ(1,1)]
    ]
    
    # ===== Cream cheese: thin layer, inset ~0.2 units, non-uniform scaling from bread =====
    V_cheese = [
        [QQ(-8,5),   QQ(-33,25), QQ(101,100)],
        [QQ(9,4),    QQ(-4,5),   QQ(101,100)],
        [QQ(-9,25),  QQ(42,25),  QQ(101,100)],
        [QQ(-8,5),   QQ(-33,25), QQ(27,25)],
        [QQ(9,4),    QQ(-4,5),   QQ(27,25)],
        [QQ(-9,25),  QQ(42,25),  QQ(27,25)]
    ]
    
    # ===== Smoked salmon: triangular base, irregular hand-set upper vertices =====
    V_salmon = [
        [QQ(-3,2),   QQ(-23,20), QQ(28,25)],
        [QQ(21,10),  QQ(-1,2),   QQ(28,25)],
        [QQ(-1,4),   QQ(31,20),  QQ(28,25)],
        [QQ(-23,20), QQ(-11,20), QQ(131,100)],
        [QQ(3,4),    QQ(-17,20), QQ(31,25)],
        [QQ(1,20),   QQ(3,4),    QQ(27,20)]
    ]
    P = convex_hull(QQ, V_bread)
    Q = convex_hull(QQ, V_salmon)
    T = convex_hull(QQ, V_cheese)

    area_P = areaofpolytope(V_bread)
    area_Q = areaofpolytope(V_salmon)
    area_T = areaofpolytope(V_cheese)

    println("Area of bread = ", area_P, " ≈ ", Float64(area_P))
    println("Area of salmon = ", area_Q, " ≈ ", Float64(area_Q))
    println("Area of cheese = ", area_T, " ≈ ", Float64(area_T))
    println("Half area of bread = ", area_P // 2, " ≈ ", Float64(area_P // 2))
    println("Half area of salmon = ", area_Q // 2," ≈ ", Float64(area_Q // 2))
    println("Half area of cheese = ", area_T // 2," ≈ ", Float64(area_T // 2))

    solutions = ham_sandwich(P,Q,T)
    println("Ham sandwich solutions: ", solutions)

    for (i, sol) in enumerate(solutions)
        a0 = QQ(rationalize(sol[1]))
        b0 = QQ(rationalize(sol[2]))
        c0 = QQ(rationalize(sol[3]))
        d0 = QQ(rationalize(sol[4]))
        cut_P = negative_areaofcut(V_bread, [a0, b0, c0,d0])
        cut_Q = negative_areaofcut(V_salmon, [a0, b0, c0,d0])
        cut_T = negative_areaofcut(V_cheese, [a0, b0, c0,d0])
        println("\nSolution $i: a=$a0, b=$b0, c=$c0, d =$d0")
        println("  Area cut of P = ", Float64(cut_P), " (target = ", area_P//2," ≈ ",  Float64(area_P//2),")")
        println("  Area cut of Q = ", Float64(cut_Q), " (target = ", area_Q//2," ≈ ",  Float64(area_Q//2),")")
        println("  Area cut of T = ", Float64(cut_T), " (target = ", area_T//2," ≈ ",  Float64(area_T//2),")")
    end
end
main()

Tracking 144 paths... 100%|████████████████████████████████████████| Time: 0:00:00
                   # paths tracked: 144
   # non-singular solutions (real): 144 (40)
       # singular endpoints (real): 0 (0)
          # total solutions (real): 144 (40)
Ham sandwich solutions: Any[[0.5684274333163554, 0.8157935855924687, 0.10663526039525538, 0.15435013300364056]]

Solution 1: a=38059264//66955361, b=103391363//126737161, c=8466824//79399853, d =30160467//195402922
  Area cut of P = 0.7055 (target = 1411//2000 ≈ 0.7055)
  Area cut of Q = 0.30041666666666667 (target = 721//2400 ≈ 0.30041666666666667)
  Area cut of T = 0.19084099999999998 (target = 190841//1000000 ≈ 0.190841)
